In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

In [6]:
csv_path = r"C:\Users\tomas\Desktop\github\upp_csde\metody_si\projekt\dane\merged_data.csv"
df = pd.read_csv(csv_path, encoding="utf-8")

df["DATA"] = pd.to_datetime(dict(year=df["ROK"], month=df["MC"], day=df["DZ"]))
df = df.sort_values(["POST", "DATA"]).reset_index(drop=True)

df["TARGET"] = df.groupby("POST")["STD"].shift(-1)
df["STD_lag1"] = df.groupby("POST")["STD"].shift(1)
df["TMAX_lag1"] = df.groupby("POST")["TMAX"].shift(1)
df["TMIN_lag1"] = df.groupby("POST")["TMIN"].shift(1)

df = pd.get_dummies(df, columns=["POST"], drop_first=False)
loc_cols = [c for c in df.columns if c.startswith("POST_")]

df = df.dropna(subset=["STD_lag1", "TMAX_lag1", "TMIN_lag1", "TARGET"])
if df.empty:
    raise ValueError("No data left after creating lags / dropping NA. Check input data.")

feature_cols = ["STD_lag1", "TMAX_lag1", "TMIN_lag1"] + loc_cols
# ensure features are numeric (convert booleans -> 0/1 and any object -> float)
X = df[feature_cols].values
X = X.astype(float)
y = df["TARGET"].values
y = y.astype(float)

split = int(len(df) * 0.8)
if split < 1:
    split = 1 if len(df) > 1 else len(df)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

if X_train.shape[0] == 0:
    raise ValueError("Training set is empty after split. Increase data or adjust split ratio.")
if X_test.shape[0] == 0:
    if X_train.shape[0] > 1:
        X_test = X_train[-1:].copy()
        y_test = y_train[-1:].copy()
        X_train = X_train[:-1]
        y_train = y_train[:-1]

        X_train = X_train.astype(float)
        X_test = X_test.astype(float)
        X_train = X_train[:-1]
        y_train = y_train[:-1]
    else:
        raise ValueError("Not enough data for testing. Need at least 2 samples.")

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std_fixed = np.where(std == 0, 1.0, std)
X_train = (X_train - mean) / std_fixed
X_test = (X_test - mean) / std_fixed

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=30,
    restore_best_weights=True
)

val_split = 0.2 if X_train.shape[0] >= 10 else 0.0
fit_kwargs = dict(
    epochs=200,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)
if val_split > 0:
    fit_kwargs.update(dict(validation_split=val_split))
else:
    fit_kwargs.update(dict(validation_data=(X_test, y_test)))

history = model.fit(X_train, y_train, **fit_kwargs)

y_pred = model.predict(X_test).flatten()

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print('MAE:', round(mae, 3))
print('RMSE:', round(rmse, 3))

wyniki = pd.DataFrame({
    'data': df['DATA'].iloc[split:split + len(y_test)].values,
    'POST': df['POST'].iloc[split:split + len(y_test)].values,
    'rzeczywiste': y_test,
    'przewidywane': y_pred
})

print(wyniki.head(20))

model.save('temperature_model2.keras')

Epoch 1/200
2656/2656 [==============================] - 7s 2ms/step - loss: 15.1910 - mae: 2.9255 - val_loss: 52.6851 - val_mae: 6.0896
Epoch 2/200
2656/2656 [==============================] - 6s 2ms/step - loss: 11.4887 - mae: 2.6425 - val_loss: 51.4905 - val_mae: 6.0380
Epoch 3/200
2656/2656 [==============================] - 6s 2ms/step - loss: 11.2995 - mae: 2.6170 - val_loss: 47.9564 - val_mae: 5.8349
Epoch 4/200
2656/2656 [==============================] - 6s 2ms/step - loss: 11.1943 - mae: 2.6023 - val_loss: 47.1708 - val_mae: 5.8077
Epoch 5/200
2656/2656 [==============================] - 6s 2ms/step - loss: 11.1285 - mae: 2.5930 - val_loss: 45.4589 - val_mae: 5.7109
Epoch 6/200
2656/2656 [==============================] - 5s 2ms/step - loss: 11.0932 - mae: 2.5885 - val_loss: 44.5980 - val_mae: 5.6651
Epoch 7/200
2656/2656 [==============================] - 6s 2ms/step - loss: 11.0652 - mae: 2.5842 - val_loss: 43.8299 - val_mae: 5.6244
Epoch 8/200
2656/2656 [==================

KeyError: 'POST'

In [7]:
model.save('temperature_model2.keras')